In [6]:
# 셀 1: 패키지 설치
!pip install -q fastapi uvicorn httpx python-multipart
!pip install -q shap lightgbm joblib pandas
!pip install -q ultralytics pytorch-grad-cam
!pip install -q google-genai
!pip install -q pyngrok
print('설치 완료!')

ERROR: Ignored the following versions that require a different python version: 8.0.10 Requires-Python >=3.7,<=3.11; 8.0.11 Requires-Python >=3.7,<=3.11; 8.0.12 Requires-Python >=3.7,<=3.11; 8.0.13 Requires-Python >=3.7,<=3.11; 8.0.14 Requires-Python >=3.7,<=3.11; 8.0.15 Requires-Python >=3.7,<=3.11; 8.0.16 Requires-Python >=3.7,<=3.11; 8.0.17 Requires-Python >=3.7,<=3.11; 8.0.18 Requires-Python >=3.7,<=3.11; 8.0.19 Requires-Python >=3.7,<=3.11; 8.0.20 Requires-Python >=3.7,<=3.11; 8.0.21 Requires-Python >=3.7,<=3.11; 8.0.22 Requires-Python >=3.7,<=3.11; 8.0.23 Requires-Python >=3.7,<=3.11; 8.0.24 Requires-Python >=3.7,<=3.11; 8.0.25 Requires-Python >=3.7,<=3.11; 8.0.26 Requires-Python >=3.7,<=3.11; 8.0.27 Requires-Python >=3.7,<=3.11; 8.0.28 Requires-Python >=3.7,<=3.11; 8.0.29 Requires-Python >=3.7,<=3.11; 8.0.30 Requires-Python >=3.7,<=3.11; 8.0.31 Requires-Python >=3.7,<=3.11; 8.0.32 Requires-Python >=3.7,<=3.11; 8.0.33 Requires-Python >=3.7,<=3.11; 8.0.34 Requires-Python >=3.7,<=3.

In [7]:
!pip install grad-cam
!pip install ultralytics
import importlib
import ultralytics
print("ultralytics 버전:", ultralytics.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 41.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=185d195b5a1439ca0ecb69259c069c3f02401e2a70063361ddad7a3e545c6c71
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam
  Using cached ultralytics-8.4.60-py3-none-any.whl.metadata (41 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics 버전: 8.4.60


In [2]:
# 셀 2: 파일 업로드
from google.colab import files

uploaded = files.upload()
print('\n업로드된 파일:', list(uploaded.keys()))

KeyboardInterrupt: 

In [8]:
# 셀 3: 서버 실행
import subprocess
import threading
import os

GEMINI_API_KEY = ''  # 여기에 Gemini API 키 입력(구글 AI 스튜디오에서 발급 가능함)

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
os.environ['YOLO_MODEL_PATH'] = 'best.pt'

# 발전량 서버 (포트 8000)
def run_power_server():
    subprocess.run(['python', '-m', 'uvicorn', 'power_xai_engine:app', '--port', '8000', '--host', '0.0.0.0'])

# XAI 라우터 (포트 8001)
def run_xai_router():
    subprocess.run(['python', '-m', 'uvicorn', 'xai_router:app', '--port', '8001', '--host', '0.0.0.0'])

threading.Thread(target=run_power_server, daemon=True).start()
threading.Thread(target=run_xai_router, daemon=True).start()

import time
time.sleep(5)
print('서버 시작 완료!')

서버 시작 완료!


In [9]:
# ngrok으로 외부 주소 생성
from pyngrok import ngrok, conf

NGROK_TOKEN = ''  # ← https://dashboard.ngrok.com 에서 발급
conf.get_default().auth_token = NGROK_TOKEN

# XAI 라우터 외부 주소 (프론트팀에 이 주소 전달)
xai_url = ngrok.connect(8001)
print('=' * 50)
print('프론트팀에 전달할 XAI 서버 주소:')
print(f'  {xai_url}')
print('=' * 50)
print()
print('API 엔드포인트:')
print(f'  POST {xai_url}/xai/dashboard')
print(f'  POST {xai_url}/xai/anomaly/xai')
print(f'  POST {xai_url}/xai/anomaly/detail')
print(f'  POST {xai_url}/xai/chat/turn')
print(f'  POST {xai_url}/xai/power/forecast')

프론트팀에 전달할 XAI 서버 주소:
  NgrokTunnel: "https://clumsily-munchkin-refuse.ngrok-free.dev" -> "http://localhost:8001"

API 엔드포인트:
  POST NgrokTunnel: "https://clumsily-munchkin-refuse.ngrok-free.dev" -> "http://localhost:8001"/xai/dashboard
  POST NgrokTunnel: "https://clumsily-munchkin-refuse.ngrok-free.dev" -> "http://localhost:8001"/xai/anomaly/xai
  POST NgrokTunnel: "https://clumsily-munchkin-refuse.ngrok-free.dev" -> "http://localhost:8001"/xai/anomaly/detail
  POST NgrokTunnel: "https://clumsily-munchkin-refuse.ngrok-free.dev" -> "http://localhost:8001"/xai/chat/turn
  POST NgrokTunnel: "https://clumsily-munchkin-refuse.ngrok-free.dev" -> "http://localhost:8001"/xai/power/forecast


In [11]:
# 셀 5: 서버 상태 확인
import requests

try:
    r = requests.get('http://localhost:8000')
    print('발전량 서버(8000):', r.json())
except:
    print('발전량 서버(8000): 아직 시작 중...')

try:
    r = requests.get('http://localhost:8001')
    print('XAI 라우터(8001):', r.json())
except:
    print('XAI 라우터(8001): 아직 시작 중...')

발전량 서버(8000): {'message': 'SolarWise AI Server Running', 'feature_count': 31}
XAI 라우터(8001): {'status': 'running', 'endpoints': ['POST /xai/dashboard', 'POST /xai/anomaly/xai', 'POST /xai/anomaly/detail', 'POST /xai/chat/turn', 'POST /xai/power/forecast']}


In [5]:
#만약 오류 떴을 때 오류 감지 코드
import subprocess
result = subprocess.run(
    ['python', '-m', 'uvicorn', 'xai_router:app', '--port', '8001', '--host', '0.0.0.0'],
    capture_output=True, text=True, timeout=15
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

STDOUT: 
STDERR: Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/__main__.py", line 4, in <module>
    uvicorn.main()
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1524, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1445, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1308, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 877, in invoke
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 441, in main
    run